# Synthetic CAN Validation — XGBoost vs CCG vs Ground Truth

Simulates two separate continuous attractor networks (CANs) of grid cells
with known ground-truth connectivity, then tests:
1. **Pairwise XGBoost** at two resolutions (100 ms / 10 ms bins; 5 ms / 1 ms bins)
2. **Cross-correlograms** for putative monosynaptic connections
3. **Comparison** of detected vs. ground-truth connections

**Networks:**
- Network A: 10 cells, 40 cm grid spacing, phase-coupled internally
- Network B: 10 cells, 60 cm grid spacing, phase-coupled internally
- 3 designated monosynaptic connections (ground truth, short latency 1–3 ms)
- Networks otherwise independent

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import LinearSegmentedColormap
from scipy.signal import fftconvolve
from scipy.ndimage import gaussian_filter
import warnings; warnings.filterwarnings('ignore')
%matplotlib inline
plt.rcParams['font.family'] = 'Arial'

fig_path = '/Users/harryclark/Documents/spatial-manifolds/scripts/figures/figure_validations/'

rng = np.random.default_rng(42)

# ── Simulation parameters ─────────────────────────────────────────────────────
ARENA_SIZE   = 100.0    # cm
T_S          = 600.0    # session length (seconds)
SPEED        = 20.0     # cm/s
N_PER_NET    = 10       # cells per network
SPACING_A    = 40.0     # grid spacing network A (cm)
SPACING_B    = 60.0     # grid spacing network B (cm)
PEAK_RATE    = 8.0      # Hz peak firing rate
BASELINE_R   = 0.5      # Hz baseline firing rate

# Phase-structured recurrent coupling (Mexican hat on grid-phase torus)
# W_ij = W_EXC·exp(−d²/2σ_exc²) − W_INH·exp(−d²/2σ_inh²)  for same-network pairs
# d is the toroidal distance between the two cells' grid phases.
# Similar-phase cells excite; dissimilar-phase cells inhibit.
TAU_WITHIN     = 30.0e-3  # PSP decay time constant (30 ms)
SIGMA_EXC_FRAC = 0.30     # excitatory σ as fraction of grid spacing
SIGMA_INH_FRAC = 0.70     # inhibitory σ as fraction of grid spacing
W_EXC          = 4.0      # peak excitatory weight (Hz)
W_INH          = 2.0      # inhibitory amplitude (Hz)

# Monosynaptic connections: (pre, post, latency_ms, weight)
# pre/post are global cell indices (0-9 = net A, 10-19 = net B)
MONO_CONNS   = [
    (2,  7,  1.5, 5.0),   # A→A: cell 2 drives cell 7 with 1.5ms latency
    (14, 3,  2.0, 4.0),   # B→A: cell 14 drives cell 3 with 2.0ms latency (cross-network)
    (1,  11, 3.0, 3.5),   # A→B: cell 1 drives cell 11 with 3.0ms latency
]

N_CELLS = N_PER_NET * 2
print(f'Total cells: {N_CELLS}  |  Session: {T_S:.0f}s')
print(f'Monosynaptic connections: {[(p,q,lat) for p,q,lat,_ in MONO_CONNS]}')

## 1. Simulate trajectory and grid firing rates

In [ ]:
# ── Trajectory: correlated random walk with proper wall reflection ─────────────
# Direction drifts with σ=0.20 rad/step (~11°/step; correlation time ~25 steps).
# Starting from a random position avoids the centre-bias from (50,50) starts.
# Wall reflection uses proper mirroring: new_x = 2*L - new_x, so position never
# piles up exactly on the boundary. Each wall handled separately (if / elif) so
# corner hits don't double-flip the angle.

DT      = 0.01          # 10 ms time steps
N_STEPS = int(T_S / DT)
t_arr   = np.arange(N_STEPS) * DT
SIGMA_ANGLE = 0.20      # rad per step

px, py = np.zeros(N_STEPS), np.zeros(N_STEPS)
px[0]  = rng.uniform(10, 90)
py[0]  = rng.uniform(10, 90)
angle  = rng.uniform(0, 2 * np.pi)

for step in range(1, N_STEPS):
    angle += rng.normal(0, SIGMA_ANGLE)
    new_x  = px[step-1] + SPEED * DT * np.cos(angle)
    new_y  = py[step-1] + SPEED * DT * np.sin(angle)
    if new_x < 0:
        new_x = -new_x;                  angle = np.pi - angle
    elif new_x > ARENA_SIZE:
        new_x = 2 * ARENA_SIZE - new_x;  angle = np.pi - angle
    if new_y < 0:
        new_y = -new_y;                   angle = -angle
    elif new_y > ARENA_SIZE:
        new_y = 2 * ARENA_SIZE - new_y;   angle = -angle
    px[step], py[step] = new_x, new_y

# ── Coverage diagnostics ──────────────────────────────────────────────────────
N_BINS_CHK = 25
edges_chk  = np.linspace(0, ARENA_SIZE, N_BINS_CHK + 1)
occ_chk, _, _ = np.histogram2d(px, py, bins=[edges_chk, edges_chk])
n_unvisited   = int((occ_chk == 0).sum())
cv_occ        = float(occ_chk[occ_chk > 0].std() / occ_chk[occ_chk > 0].mean())
print(f'Trajectory: {N_STEPS} steps  ({T_S:.0f} s at {SPEED} cm/s)')
print(f'Coverage : {N_BINS_CHK**2 - n_unvisited}/{N_BINS_CHK**2} bins visited '
      f'({100*(1 - n_unvisited/N_BINS_CHK**2):.1f}%)')
print(f'Occupancy CV = {cv_occ:.2f}  (lower → more uniform; <0.6 is good)')

# ── Grid firing rate: Gaussian bumps at hexagonal lattice points ─────────────
def hex_grid_rate(px, py, spacing, phase_x, phase_y,
                  sigma=None, peak=PEAK_RATE, base=BASELINE_R):
    if sigma is None:
        sigma = spacing / 5.0
    a1 = np.array([spacing,           0.0])
    a2 = np.array([spacing / 2.0, spacing * np.sqrt(3) / 2.0])
    rate    = np.zeros(len(px))
    n_range = int(np.ceil(ARENA_SIZE / spacing)) + 2
    for ii in range(-n_range, n_range + 1):
        for jj in range(-n_range, n_range + 1):
            pt = ii * a1 + jj * a2 + np.array([phase_x % spacing,
                                                phase_y % spacing])
            d2 = (px - pt[0])**2 + (py - pt[1])**2
            rate += np.exp(-d2 / (2.0 * sigma**2))
    r_max = rate.max()
    if r_max > 0:
        rate /= r_max
    return base + (peak - base) * rate

# ── Pre-compute rate maps on fine grid, then interpolate per trajectory step ──
from scipy.interpolate import RegularGridInterpolator

FINE   = 200
xs_f   = np.linspace(0, ARENA_SIZE, FINE)
ys_f   = np.linspace(0, ARENA_SIZE, FINE)
XX_f, YY_f = np.meshgrid(xs_f, ys_f)
px_f, py_f = XX_f.ravel(), YY_f.ravel()

phases_A = rng.uniform(0, SPACING_A, size=(N_PER_NET, 2))
phases_B = rng.uniform(0, SPACING_B, size=(N_PER_NET, 2))

print('Pre-computing grid rate maps...')
base_rates = np.zeros((N_CELLS, N_STEPS))
for ci in range(N_PER_NET):
    for net_i, (spacing, phases) in enumerate([(SPACING_A, phases_A),
                                               (SPACING_B, phases_B)]):
        idx     = ci + net_i * N_PER_NET
        rm_flat = hex_grid_rate(px_f, py_f, spacing, *phases[ci])
        rm_2d   = rm_flat.reshape(FINE, FINE)   # (y, x)
        interp  = RegularGridInterpolator(
            (ys_f, xs_f), rm_2d,
            method='linear', bounds_error=False, fill_value=BASELINE_R)
        base_rates[idx] = interp(np.column_stack([py, px]))
    if ci == 0:
        print(f'  Rate range cell 0: [{base_rates[0].min():.2f}, '
              f'{base_rates[0].max():.2f}] Hz')

print(f'Mean peak rate network A: {base_rates[:N_PER_NET].max(axis=1).mean():.2f} Hz')
print(f'Mean peak rate network B: {base_rates[N_PER_NET:].max(axis=1).mean():.2f} Hz')

## 2. Simulate spike trains with network coupling

In [ ]:
# ── Phase-structured recurrent weight matrix ──────────────────────────────────
# Distance between cells i and j is their toroidal phase distance —
# i.e. how far apart their grid phases are on the unit-cell torus.
# W_ij follows a Mexican hat: short-range excitation, longer-range inhibition.
# Cross-network pairs are left at 0 (handled only by MONO_CONNS).

def _torus_dist(phi_i, phi_j, spacing):
    dx = abs(float(phi_i[0]) - float(phi_j[0]))
    dy = abs(float(phi_i[1]) - float(phi_j[1]))
    dx = min(dx, spacing - dx)
    dy = min(dy, spacing - dy)
    return np.sqrt(dx**2 + dy**2)

W_net = np.zeros((N_CELLS, N_CELLS))
for i in range(N_CELLS):
    net_i    = i // N_PER_NET
    spacing  = SPACING_A if net_i == 0 else SPACING_B
    phases   = phases_A  if net_i == 0 else phases_B
    sig_exc  = SIGMA_EXC_FRAC * spacing
    sig_inh  = SIGMA_INH_FRAC * spacing
    ci       = i % N_PER_NET
    for j in range(N_CELLS):
        if i == j: continue
        if (j // N_PER_NET) != net_i: continue
        cj = j % N_PER_NET
        d  = _torus_dist(phases[ci], phases[cj], spacing)
        W_net[i, j] = (W_EXC * np.exp(-d**2 / (2 * sig_exc**2))
                       - W_INH * np.exp(-d**2 / (2 * sig_inh**2)))

w_vals = W_net[W_net != 0]
print(f'Phase-structured W_net:')
print(f'  Excitatory pairs (W>0): {(w_vals > 0).sum()}  '
      f'mean={w_vals[w_vals>0].mean():.2f} Hz  max={w_vals.max():.2f} Hz')
print(f'  Inhibitory pairs (W<0): {(w_vals < 0).sum()}  '
      f'mean={w_vals[w_vals<0].mean():.2f} Hz  min={w_vals.min():.2f} Hz')

# ── Spike generation ──────────────────────────────────────────────────────────
alpha_within = np.exp(-DT / TAU_WITHIN)

mono_steps   = [(p, q, max(1, int(lat_ms / (DT * 1000))), w)
                for p, q, lat_ms, w in MONO_CONNS]
max_mono_lag = max(s for _, _, s, _ in mono_steps)

spike_trains = np.zeros((N_CELLS, N_STEPS), dtype=np.int8)
psp          = np.zeros(N_CELLS)
spike_buffer = np.zeros((N_CELLS, max_mono_lag + 1), dtype=np.int8)
buf_ptr      = 0

print('Simulating spike trains...')
for step in range(N_STEPS):
    rate = base_rates[:, step] + psp.copy()
    for pre, post, lag, weight in mono_steps:
        past_ptr  = (buf_ptr - lag) % (max_mono_lag + 1)
        rate[post] += weight * spike_buffer[pre, past_ptr]
    rate   = np.maximum(rate, 0)
    spikes = (rng.random(N_CELLS) < rate * DT).astype(np.int8)
    spike_trains[:, step] = spikes
    spike_buffer[:, buf_ptr] = spikes
    buf_ptr = (buf_ptr + 1) % (max_mono_lag + 1)
    psp     = psp * alpha_within + W_net @ spikes.astype(float)
    if step % 10000 == 0:
        print(f'  step {step}/{N_STEPS}', end='\r')

print(f'\nDone.  Total spikes: {spike_trains.sum():,}')
fr = spike_trains.sum(axis=1) / T_S
for i in range(N_CELLS):
    net = 'A' if i < N_PER_NET else 'B'
    print(f'  Cell {i:2d} (Net {net}): {fr[i]:.2f} Hz')

In [ ]:
# ── Visualise the phase-structured weight matrix ──────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(13, 4))

im0 = axes[0].imshow(W_net, cmap='RdBu_r', vmin=-W_EXC, vmax=W_EXC,
                      interpolation='nearest', aspect='auto')
axes[0].axhline(N_PER_NET - 0.5, color='k', lw=1.2)
axes[0].axvline(N_PER_NET - 0.5, color='k', lw=1.2)
plt.colorbar(im0, ax=axes[0], label='Weight (Hz)')
axes[0].set_title('Full W_net\n(black lines = network boundary)', fontsize=9)
axes[0].set_xlabel('Post-synaptic cell'); axes[0].set_ylabel('Pre-synaptic cell')

# W_net for network A only, sorted by phase_x
idx_A   = np.arange(N_PER_NET)
sort_A  = idx_A[np.argsort(phases_A[:, 0])]
W_A_sorted = W_net[np.ix_(sort_A, sort_A)]
im1 = axes[1].imshow(W_A_sorted, cmap='RdBu_r', vmin=-W_EXC, vmax=W_EXC,
                      interpolation='nearest', aspect='auto')
plt.colorbar(im1, ax=axes[1], label='Weight (Hz)')
axes[1].set_title(f'Net A  (sorted by phase_x)\nσ_exc={SIGMA_EXC_FRAC*SPACING_A:.0f}cm  '
                  f'σ_inh={SIGMA_INH_FRAC*SPACING_A:.0f}cm', fontsize=9)
axes[1].set_xlabel('Cell (sorted by phase)'); axes[1].set_ylabel('Cell (sorted by phase)')

# Mexican hat profile vs phase distance for net A
d_range = np.linspace(0, SPACING_A, 200)
sig_exc_A = SIGMA_EXC_FRAC * SPACING_A
sig_inh_A = SIGMA_INH_FRAC * SPACING_A
w_profile = (W_EXC * np.exp(-d_range**2 / (2*sig_exc_A**2))
             - W_INH * np.exp(-d_range**2 / (2*sig_inh_A**2)))
axes[2].plot(d_range, w_profile, 'k-', lw=2)
axes[2].axhline(0, color='grey', lw=0.8, ls='--')
axes[2].axvline(SPACING_A / 2, color='#888', lw=0.8, ls=':', label='half-spacing')
axes[2].fill_between(d_range, w_profile, 0,
                      where=w_profile > 0, color='#c04744', alpha=0.3, label='excitatory')
axes[2].fill_between(d_range, w_profile, 0,
                      where=w_profile < 0, color='#3171ae', alpha=0.3, label='inhibitory')
axes[2].set_xlabel('Phase distance (cm)', fontsize=9)
axes[2].set_ylabel('Weight (Hz)', fontsize=9)
axes[2].set_title('Mexican hat profile\n(Net A, spacing=40cm)', fontsize=9)
axes[2].legend(fontsize=8, frameon=False)
axes[2].spines[['top', 'right']].set_visible(False)

plt.suptitle('Phase-structured recurrent weights', fontsize=10, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Visualise: grid rate maps and example spike trains

In [ ]:
# ── 2D rate maps from simulated spikes ───────────────────────────────────────
N_BINS = 25
edges  = np.linspace(0, ARENA_SIZE, N_BINS + 1)

def make_rate_map(spike_train, px, py):
    tc,  _, _ = np.histogram2d(px, py, bins=[edges, edges], weights=spike_train.astype(float))
    occ, _, _ = np.histogram2d(px, py, bins=[edges, edges])
    rm = tc / np.where(occ * DT > 0, occ * DT, np.nan)
    return gaussian_filter(np.nan_to_num(rm).T, sigma=1.2)

rate_maps = [make_rate_map(spike_trains[i], px, py) for i in range(N_CELLS)]

# Plot: 4 example rate maps per network + spike rasters
fig, axes = plt.subplots(3, N_PER_NET, figsize=(N_PER_NET*1.8, 5.5),
                          gridspec_kw={'hspace': 0.35, 'wspace': 0.10})
COL_A, COL_B = '#c04744', '#3171ae'

for i in range(N_PER_NET):
    for row, net_i in enumerate([i, N_PER_NET+i]):
        ax  = axes[row, i]
        col = COL_A if net_i < N_PER_NET else COL_B
        vmax = np.nanpercentile(rate_maps[net_i], 99)
        ax.imshow(rate_maps[net_i], origin='lower', cmap='viridis',
                  vmin=0, vmax=vmax, interpolation='nearest')
        ax.set_xticks([]); ax.set_yticks([])
        ax.set_title(f'C{net_i}\n{fr[net_i]:.1f}Hz', fontsize=7, color=col)

# Spike rasters (first 10 seconds)
T_SHOW = int(10 / DT)
t_show = t_arr[:T_SHOW]
for i in range(N_PER_NET):
    axes[2, i].axis('off')
for net_i in range(N_CELLS):
    col_i = net_i % N_PER_NET
    offset = net_i if net_i < N_PER_NET else net_i - N_PER_NET
    ax = axes[2, 0]
    spike_t = t_show[spike_trains[net_i, :T_SHOW] > 0]
    col = COL_A if net_i < N_PER_NET else COL_B
    ax.vlines(spike_t, net_i + 0.1, net_i + 0.9, color=col, lw=0.5, alpha=0.7)

ax = axes[2, 0]
ax.set_visible(True); ax.set_xlim(0, 10); ax.set_ylim(-0.5, N_CELLS)
ax.set_xlabel('Time (s)', fontsize=8); ax.set_ylabel('Cell', fontsize=8)
ax.axhline(N_PER_NET - 0.5, color='k', lw=0.8, ls='--', alpha=0.5)
ax.spines[['top','right']].set_visible(False); ax.tick_params(labelsize=7)

# Mark monosynaptic pairs
for pre, post, _, _ in MONO_CONNS:
    ax.annotate('', xy=(10.1, post+0.5), xytext=(10.1, pre+0.5),
                arrowprops=dict(arrowstyle='->', color='#e67e22', lw=1.2),
                annotation_clip=False)

axes[0, 0].set_title(f'Network A (spacing={SPACING_A}cm)  C0\n{fr[0]:.1f}Hz',
                      fontsize=7, color=COL_A)
axes[1, 0].set_title(f'Network B (spacing={SPACING_B}cm)  C{N_PER_NET}\n{fr[N_PER_NET]:.1f}Hz',
                      fontsize=7, color=COL_B)
fig.suptitle('Simulated grid cells — rate maps and spike rasters (orange arrows = ground-truth connections)',
             fontsize=9, fontweight='bold')
fig.savefig(fig_path + 'synthetic_rate_maps.pdf', bbox_inches='tight', dpi=200)
plt.show()

## 4. Pairwise XGBoost at two time resolutions

In [ ]:
from spatial_manifolds.mlencoding import MLencoding

# Position normalised to [0, 1] at DT=10ms resolution (matches spike bins)
pos_x = px / ARENA_SIZE
pos_y = py / ARENA_SIZE

CONFIGS = {
    '100ms / 10ms bins': dict(bin_ms=10, history_ms=100, n_filters=5),
    '5ms / 1ms bins':    dict(bin_ms=1,  history_ms=5,   n_filters=5),
}
N_CV = 5

pr2_matrices = {}   # config_label → (N_CELLS, N_CELLS) Δpρ² matrix

for config_label, cfg in CONFIGS.items():
    print(f'\n── {config_label} ──')
    bin_ms = cfg['bin_ms']
    factor = max(1, 10 // bin_ms)

    # Re-bin spikes and position to requested resolution
    spk_binned = np.repeat(spike_trains, factor, axis=1).astype(float) / factor
    pos_x_bin  = np.repeat(pos_x, factor)[:spk_binned.shape[1]]
    pos_y_bin  = np.repeat(pos_y, factor)[:spk_binned.shape[1]]
    T_bin      = spk_binned.shape[1]

    # Position-only baseline covariate: (T_bin, 2)
    x_pos = np.column_stack([pos_x_bin, pos_y_bin])

    xgb = MLencoding(tunemodel='xgboost', cov_history=True, spike_history=False,
                     window=bin_ms, n_filters=cfg['n_filters'],
                     max_time=cfg['history_ms'])

    pr2_mat = np.full((N_CELLS, N_CELLS), np.nan)

    for target in range(N_CELLS):
        y = spk_binned[target]

        # Fit position-only baseline once per target
        _, pr2_pos = xgb.fit_cv(x_pos, y, verbose=0,
                                 continuous_folds=True, n_cv=N_CV)
        pr2_pos_mean = float(np.nanmean(pr2_pos))

        for cov in range(N_CELLS):
            if cov == target:
                continue
            # Full model: position + covariate spike history
            x_full = np.column_stack([x_pos, spk_binned[cov]])
            _, pr2_full = xgb.fit_cv(x_full, y, verbose=0,
                                      continuous_folds=True, n_cv=N_CV)
            pr2_mat[cov, target] = float(np.nanmean(pr2_full)) - pr2_pos_mean

        print(f'  target {target}/{N_CELLS-1}', end='\r')

    pr2_matrices[config_label] = pr2_mat
    print(f'\n  max Δpρ² (above pos) = {np.nanmax(pr2_mat):.4f}')

print('\nXGBoost done.')

## 5. Cross-correlograms

In [ ]:
# CCG at 1ms bins over ±50ms window
CCG_BIN_MS  = 1
CCG_LAG_MS  = 50
N_CCG_LAGS  = 2 * CCG_LAG_MS + 1
lags_ms     = np.arange(-CCG_LAG_MS, CCG_LAG_MS + 1)

# Re-bin spikes at 1ms for CCG
spk_1ms = np.repeat(spike_trains, 10, axis=1).astype(float) / 10
T_1ms   = spk_1ms.shape[1]

ccg_mat = np.zeros((N_CELLS, N_CELLS, N_CCG_LAGS))

for i in range(N_CELLS):
    for j in range(N_CELLS):
        if i == j: continue
        full = fftconvolve(spk_1ms[i], spk_1ms[j][::-1], mode='full')
        ctr  = T_1ms - 1
        ccg_mat[i, j] = full[ctr - CCG_LAG_MS : ctr + CCG_LAG_MS + 1]

# Normalise by geometric mean spike count
n_spk_1ms = spk_1ms.sum(axis=1)
for i in range(N_CELLS):
    for j in range(N_CELLS):
        if i == j: continue
        norm = np.sqrt(n_spk_1ms[i] * n_spk_1ms[j]) * (CCG_BIN_MS / 1000)
        ccg_mat[i, j] /= (norm + 1e-10)

print('CCGs computed.')

## 6. Comparison figure — ground truth vs XGBoost vs CCG

In [ ]:
# ── Comparison: W_net ground truth vs XGBoost Δpρ² (above position) ──────────
n_configs = len(pr2_matrices)
fig = plt.figure(figsize=(5 * (n_configs + 2), 10))
gs  = gridspec.GridSpec(2, n_configs + 2, figure=fig,
                         hspace=0.45, wspace=0.35,
                         left=0.06, right=0.97, top=0.92, bottom=0.07)

# ── Row 0, col 0: W_net ground-truth ─────────────────────────────────────────
ax_gt = fig.add_subplot(gs[0, 0])
vmax_gt = max(abs(W_net.min()), W_net.max())
im = ax_gt.imshow(W_net, cmap='RdBu_r', vmin=-vmax_gt, vmax=vmax_gt,
                   aspect='auto', interpolation='nearest')
plt.colorbar(im, ax=ax_gt, fraction=0.04).set_label('Weight (Hz)', fontsize=7)
ax_gt.axhline(N_PER_NET - 0.5, color='k', lw=1.2)
ax_gt.axvline(N_PER_NET - 0.5, color='k', lw=1.2)
ax_gt.set_title('W_net ground truth\n(phase-structured Mexican hat)', fontsize=8,
                 fontweight='bold')
ax_gt.set_xlabel('Post-synaptic'); ax_gt.set_ylabel('Pre-synaptic')
ax_gt.tick_params(labelsize=6)

# ── Row 0, cols 1+: XGBoost Δpρ² matrices ────────────────────────────────────
vmax_pr2 = max(np.nanpercentile(m, 98) for m in pr2_matrices.values())
for ci, (cfg_label, pr2_mat) in enumerate(pr2_matrices.items()):
    ax = fig.add_subplot(gs[0, ci + 1])
    im = ax.imshow(pr2_mat, cmap='hot', vmin=0, vmax=vmax_pr2,
                   aspect='auto', interpolation='nearest')
    plt.colorbar(im, ax=ax, fraction=0.04).set_label('Δpρ² above pos.', fontsize=7)
    ax.axhline(N_PER_NET - 0.5, color='white', lw=1.2)
    ax.axvline(N_PER_NET - 0.5, color='white', lw=1.2)
    ax.set_title(f'XGBoost Δpρ² (above position)\n{cfg_label}', fontsize=8,
                  fontweight='bold')
    ax.set_xlabel('Target cell'); ax.tick_params(labelsize=6)

# ── Row 0, last col: scatter W_net vs Δpρ² ───────────────────────────────────
ax_sc = fig.add_subplot(gs[0, -1])
COL_SAME  = '#c04744'   # within-network
COL_CROSS = '#888888'   # cross-network (W_net = 0)
for ci, (cfg_label, pr2_mat) in enumerate(pr2_matrices.items()):
    alpha = 0.6 if ci == 0 else 0.3
    for i in range(N_CELLS):
        for j in range(N_CELLS):
            if i == j or np.isnan(pr2_mat[i, j]):
                continue
            same_net = (i // N_PER_NET) == (j // N_PER_NET)
            col = COL_SAME if same_net else COL_CROSS
            if ci == 0:
                ax_sc.scatter(W_net[i, j], pr2_mat[i, j],
                               s=18, alpha=alpha, color=col, edgecolors='none')

# Correlation line (within-network only)
wn_pairs = [(i, j) for i in range(N_CELLS) for j in range(N_CELLS)
             if i != j and (i // N_PER_NET) == (j // N_PER_NET)]
w_vals_wn  = np.array([W_net[i, j]      for i, j in wn_pairs])
pr2_vals_wn = np.array([list(pr2_matrices.values())[0][i, j] for i, j in wn_pairs])
valid = ~np.isnan(pr2_vals_wn)
if valid.sum() > 2:
    r = np.corrcoef(w_vals_wn[valid], pr2_vals_wn[valid])[0, 1]
    ax_sc.set_title(f'W_net vs Δpρ² (above pos.)\nWithin-net r={r:.2f}', fontsize=8)
ax_sc.axhline(0, color='grey', lw=0.7, ls='--')
ax_sc.axvline(0, color='grey', lw=0.7, ls='--')
ax_sc.set_xlabel('W_net weight (Hz)', fontsize=8)
ax_sc.set_ylabel('Δpρ² above position', fontsize=8)
from matplotlib.lines import Line2D
ax_sc.legend(handles=[
    Line2D([0],[0], marker='o', color='w', markerfacecolor=COL_SAME,  ms=7, label='Within-network'),
    Line2D([0],[0], marker='o', color='w', markerfacecolor=COL_CROSS, ms=7, label='Cross-network'),
], fontsize=7, frameon=False)
ax_sc.spines[['top','right']].set_visible(False)

# ── Row 1: distribution of Δpρ² split by W_net sign ─────────────────────────
ax_dist = fig.add_subplot(gs[1, :])
colors  = {'exc. (W>0)': '#c04744', 'inh. (W<0)': '#3171ae', 'cross-net': '#888888'}
for ci, (cfg_label, pr2_mat) in enumerate(pr2_matrices.items()):
    vals = {'exc. (W>0)': [], 'inh. (W<0)': [], 'cross-net': []}
    for i in range(N_CELLS):
        for j in range(N_CELLS):
            if i == j or np.isnan(pr2_mat[i, j]):
                continue
            same_net = (i // N_PER_NET) == (j // N_PER_NET)
            if not same_net:
                vals['cross-net'].append(pr2_mat[i, j])
            elif W_net[i, j] > 0:
                vals['exc. (W>0)'].append(pr2_mat[i, j])
            else:
                vals['inh. (W<0)'].append(pr2_mat[i, j])
    offset = ci * 0.004
    for k, (label, v) in enumerate(vals.items()):
        if len(v) == 0:
            continue
        v = np.array(v)
        ax_dist.scatter([k + offset] * len(v), v, s=12, alpha=0.4,
                         color=colors[label], edgecolors='none')
        ax_dist.errorbar(k + offset, np.nanmean(v), yerr=np.nanstd(v)/np.sqrt(len(v)),
                          fmt='D', ms=6, color=colors[label],
                          capsize=3, lw=1.5,
                          label=f'{label} ({cfg_label.split("/")[0].strip()})')

ax_dist.axhline(0, color='k', lw=0.8, ls='--')
ax_dist.set_xticks([0, 1, 2])
ax_dist.set_xticklabels(['Excitatory\n(W>0)', 'Inhibitory\n(W<0)', 'Cross-network\n(W=0)'])
ax_dist.set_ylabel('Δpρ² above position', fontsize=9)
ax_dist.set_title('Δpρ² by connection type — excitatory pairs should have highest values',
                   fontsize=9)
ax_dist.legend(fontsize=7, frameon=False, ncol=2)
ax_dist.spines[['top', 'right']].set_visible(False)

fig.suptitle('Phase-structured CAN validation\n'
             'Δpρ² = pρ²(position + cell) − pρ²(position only)',
             fontsize=10, fontweight='bold')
fig.savefig(fig_path + 'synthetic_validation.pdf', bbox_inches='tight', dpi=200)
plt.show()
print('Saved.')